# Lab 13 — Dashboards, Streaming, and Deployment
## E-Commerce Product Intelligence Pipeline

## Part 1 — Install Required Libraries

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "dash>=2.17.0",
    "dash-bootstrap-components>=1.6.0",
    "pymongo>=4.6.0",
    "gunicorn>=22.0.0",
    "websockets>=12.0.0",
], check=True)
print("All packages ready.")

In [ ]:
import dash
import dash_bootstrap_components as dbc
import pymongo
import plotly
import pandas as pd

print(f"dash                      {dash.__version__}")
print(f"dash-bootstrap-components {dbc.__version__}")
print(f"pymongo                   {pymongo.__version__}")
print(f"plotly                    {plotly.__version__}")
print(f"pandas                    {pd.__version__}")

## Part 2 — Data Access Layer

All database queries are isolated in `src/dashboard/data_access.py`. This keeps callbacks clean and makes the data layer independently testable.

The module reads a `MONGO_URI` environment variable, so the same code works locally (`mongodb://localhost:27017`) and inside Docker Compose (`mongodb://db:27017`).

In [ ]:
import sys, os
sys.path.insert(0, os.path.join("..", "src"))

from dashboard.data_access import (
    load_products_df, get_sources, get_categories,
    get_rating_range, filter_products
)

df = load_products_df()
print(f"Shape  : {df.shape}")
print(f"Columns: {list(df.columns)}")
df[["title", "rating", "source", "author"]].head(3)

In [ ]:
sources = get_sources(df)
categories = get_categories(df)
rating_min, rating_max = get_rating_range(df)

print(f"Sources ({len(sources)}): {sources[:5]}...")
print(f"Categories ({len(categories)}): {categories[:5]}...")
print(f"Rating range: {rating_min} – {rating_max}")

### Demonstrating `filter_products()`
All three filter parameters are optional. Passing `None` skips that filter entirely.

In [ ]:
# Filter by source only
source_df = filter_products(df, source=sources[0] if sources else None)
print(f"Products from '{sources[0] if sources else 'N/A'}': {len(source_df)}")

# Filter by title search
searched = filter_products(df, search="ninja")
print(f"Titles containing 'ninja': {len(searched)}")
searched[["title", "rating", "source"]].head(3)

## Part 3 — Understanding the Dash Framework

Every Dash application has four ordered parts:

1. **Create the app object** — `Dash(__name__, external_stylesheets=[...])`
2. **Define the layout** — a tree of Python objects that Dash translates to HTML
3. **Register callbacks** — `@app.callback(Output(...), Input(...))` decorators
4. **Run the server** — `app.run(debug=True)` for development; Gunicorn in production

### Layout building blocks

| Module | Role |
|---|---|
| `dash.html` | One Python class per HTML tag (`html.Div`, `html.H1`, …) |
| `dash.dcc` | Higher-level interactive components (`dcc.Dropdown`, `dcc.Graph`, `dcc.Interval`) |
| `dash_bootstrap_components` | Responsive grid, cards, navbars |

### Callback anatomy
```python
@app.callback(
    Output("revenue-chart", "figure"),   # what gets updated
    Input("source-filter",  "value"),    # what triggers the update
)
def update_chart(selected_source):
    df = filter_products(_DF, source=selected_source)
    return px.bar(df, x="title", y="rating")
```
When the user changes the dropdown, Dash automatically re-runs `update_chart` and pushes the new figure to the browser — no page reload needed.

## Part 4 — Chart Demonstrations

Each chart type is justified by the nature of the data:

| Chart | Type | Justification |
|---|---|---|
| Top 10 Products by Rating | Horizontal bar | Long product titles read left-to-right (Tufte: match chart to data) |
| Rating by Source | Box plot | Shows full distribution: median, IQR, outliers — not just the mean |
| Rating Distribution | Histogram | Correct for showing frequency of a single continuous variable |
| Products per Source | Bar chart | Comparing discrete categories by a single numeric measure |

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

DARK_TEMPLATE = "plotly_dark"
CHART_BG = "#112236"

# ── Chart 1: Top 10 Products by Rating ────────────────────────────────────
top10 = (
    df.dropna(subset=["rating", "title"])
    .drop_duplicates("title")
    .nlargest(10, "rating")[["title", "rating"]]
    .sort_values("rating")
)

fig1 = px.bar(
    top10, x="rating", y="title", orientation="h",
    text=top10["rating"].map(lambda v: f"{v:.1f} ★"),
    color="rating", color_continuous_scale="Blues",
    template=DARK_TEMPLATE,
    title="Top 10 Products by Rating",
    labels={"rating": "Rating (0–5)", "title": ""},
)
fig1.update_traces(textposition="outside")
fig1.update_layout(
    paper_bgcolor=CHART_BG, plot_bgcolor=CHART_BG,
    coloraxis_showscale=False,
    yaxis={"categoryorder": "total ascending"},
)
fig1.show()

**Chart 1 — Top 10 Products by Rating:** Horizontal bar chart chosen because product titles are long strings that read naturally left-to-right. Colour encodes rating magnitude. Following Tufte's data-ink principle, the colour scale legend is hidden since the x-axis already shows the values.

In [ ]:
# ── Chart 2: Rating Distribution by Source (Box Plot) ─────────────────────
fig2 = px.box(
    df, x="source", y="rating", color="source",
    template=DARK_TEMPLATE,
    title="Rating Distribution by Source",
    labels={"rating": "Rating (0–5)", "source": ""},
    points="outliers",
)
fig2.update_layout(
    paper_bgcolor=CHART_BG, plot_bgcolor=CHART_BG, showlegend=False,
)
fig2.show()

**Chart 2 — Rating Distribution by Source:** Box plot chosen because it simultaneously shows the median, interquartile range, and outliers across different data sources. A simple bar of means would hide the variance in review scores, which is the most informative signal for comparing sources.

In [ ]:
# ── Chart 3: Rating Distribution Histogram ────────────────────────────────
fig3 = px.histogram(
    df.dropna(subset=["rating"]),
    x="rating", nbins=20,
    color_discrete_sequence=["#4a9eff"],
    template=DARK_TEMPLATE,
    title="Rating Distribution Histogram",
    labels={"rating": "Rating (0–5)", "count": "Number of Reviews"},
)
fig3.update_layout(
    paper_bgcolor=CHART_BG, plot_bgcolor=CHART_BG, bargap=0.05
)
fig3.show()

**Chart 3 — Rating Distribution Histogram:** Histogram is the correct choice for showing the frequency distribution of a single continuous variable (rating). It reveals whether reviews cluster at 5-stars (positive bias) or are more evenly spread, which is key insight for product intelligence.

In [ ]:
# ── Chart 4: Products per Source (Bar Chart) ──────────────────────────────
counts = df.groupby("source").size().reset_index(name="count")

fig4 = px.bar(
    counts, x="source", y="count",
    color="count", color_continuous_scale="Blues",
    template=DARK_TEMPLATE,
    title="Number of Reviews per Source",
    labels={"source": "Source", "count": "Number of Reviews"},
)
fig4.update_layout(
    paper_bgcolor=CHART_BG, plot_bgcolor=CHART_BG, coloraxis_showscale=False
)
fig4.show()

**Chart 4 — Products per Source:** Bar chart is correct for comparing discrete categories (source files) by a single numeric measure (count). Each bar is independent — there is no continuous relationship between sources, so a line chart would be misleading.

In [ ]:
# ── Chart 5: Live Ticker (simulated — static snapshot) ────────────────────
import random, collections

buffer = collections.deque(maxlen=60)
for i in range(20):
    buffer.append({"tick": i, "value": 4.0 + random.gauss(0, 0.3) + 0.05 * (i % 10)})

fig5 = go.Figure(go.Scatter(
    x=[p["tick"] for p in buffer],
    y=[p["value"] for p in buffer],
    mode="lines+markers",
    line=dict(color="#34d399", width=2),
    fill="tozeroy", fillcolor="rgba(52,211,153,0.1)",
))
fig5.update_layout(
    template=DARK_TEMPLATE, paper_bgcolor=CHART_BG, plot_bgcolor=CHART_BG,
    title="Live Rating Ticker (simulated — 20 ticks)",
    xaxis_title="Tick", yaxis_title="Simulated Avg Rating",
    yaxis=dict(range=[2.5, 5.5]),
    height=250,
)
fig5.show()

**Chart 5 — Live Ticker:** Line chart with fill shows simulated real-time average product rating updates. In the live dashboard this updates every 3 seconds via `dcc.Interval`. The filled area below the line encodes the cumulative signal, making trend changes immediately visible.

## Part 5 — Callback Patterns

| Pattern | Callback | Trigger | Output |
|---|---|---|---|
| N → 1 | Top products bar | source + category + search | revenue-chart figure |
| N → 1 | Rating box | source + category + search | rating-chart figure |
| N → 1 | Histogram | source + category + search | scatter-chart figure |
| N → 1 | Products per source bar | source + category + search | trend-chart figure |
| 1 → 1 | Live ticker | dcc.Interval tick | live-chart figure |

### Live Ticker Design
`dcc.Interval` fires a callback every 3000 ms. The callback appends a randomly-generated data point to a `collections.deque(maxlen=60)` module-level buffer. Because the deque has a fixed maximum length, old points are automatically discarded — no unbounded memory growth.

In a real production system the callback would read from:
- A WebSocket stream (e.g. live product review API)
- A time-series database such as InfluxDB
- A message queue such as Kafka

## Part 6 — Dockerfile Walkthrough

In [ ]:
dockerfile_path = os.path.join("..", "Dockerfile")
with open(dockerfile_path) as f:
    print(f.read())

### Dockerfile instruction explanations

| Instruction | Purpose |
|---|---|
| `FROM python:3.12-slim` | Minimal Debian image with Python 3.12 — smaller than the full image |
| `WORKDIR /app` | All subsequent commands run in `/app` inside the container |
| `COPY requirements.txt .` | Copy only the requirements file first to enable layer caching |
| `RUN pip install --no-cache-dir` | Install deps; `--no-cache-dir` keeps the image smaller |
| `COPY . .` | Copy application code (separate layer from deps) |
| `EXPOSE 8050` | Documents the port (does not actually publish it) |
| `CMD ["gunicorn", ...]` | Production WSGI server — multi-worker, no hot-reload |

### Why Gunicorn instead of `app.run()`?
`app.run(debug=True)` runs a single-threaded Flask development server with hot-reload enabled. Gunicorn forks multiple worker processes and has no development overhead — it is the correct production choice.

## Part 7 — Running the Dashboard Locally

In [ ]:
import importlib.util, pathlib

app_path = pathlib.Path("..") / "app.py"
spec = importlib.util.spec_from_file_location("app", app_path)
app_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(app_module)

print("app.py imported successfully.")
print(f"Dash app title : {app_module.app.title}")
print(f"Flask server   : {type(app_module.server).__name__}")

To start the dashboard in development mode, run from the project root:
```
python app.py
```
Then open **http://127.0.0.1:8050** in a browser.

## Part 8 — MongoDB Seeding

In [ ]:
seed_path = os.path.join("..", "scripts", "seed_mongo.py")
with open(seed_path) as f:
    print(f.read())

In [ ]:
import subprocess
result = subprocess.run(
    [sys.executable, seed_path],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("[Note] MongoDB not running — dashboard will fall back to CSV.")
    print(result.stderr[:300])

## Part 9 — Deployment Documentation

### Prerequisites
- **Docker Desktop** (v4.x or later) installed and running on your machine.
- **Ports 8050 and 27017** not in use by other applications.
- The project repository cloned locally.
- Python 3.11+ and pip available (only needed for the seed script and local dev).

### Step-by-Step Deployment

**Step 1: Build and start the full stack**
```bash
docker compose up --build
```
This command:
- Builds the Dash app image from the local `Dockerfile`.
- Pulls the `mongo:7` image from Docker Hub.
- Starts both services on a private Docker network.

Wait until you see `Booting worker with pid` in the logs — the app is then ready.

To run in the background (detached mode):
```bash
docker compose up --build -d
docker compose logs -f
```

**Step 2: Seed MongoDB (first run only)**
```bash
python scripts/seed_mongo.py
```
This loads `data/processed/analytics/products_raw.csv` into the `ecommerce_db.products` MongoDB collection and creates indexes on `source`, `category`, `title`, and `rating`.

Run this only once. Subsequent starts do not need re-seeding unless you run `docker compose down -v`.

**Step 3: Open the dashboard**

Navigate to **http://localhost:8050** in any browser.

**Step 4: Verify functionality**
- KPI cards display total reviews, average rating, unique sources, and unique categories.
- Change the **Source** dropdown — all four charts update instantly.
- Change the **Category** dropdown — charts filter to that category.
- Type a title in the **Search** box — charts filter to matching products.
- The **Live Ticker** updates automatically every 3 seconds.

**Step 5: Stop the stack**
```bash
docker compose down        # stop containers, keep database volume
docker compose down -v     # also delete the mongo-data volume
```

### Troubleshooting

| Problem | Solution |
|---|---|
| Port 8050 already in use | Change host port in `docker-compose.yml`: `"8051:8050"` |
| Port 27017 already in use | Change: `"27018:27017"` and update `MONGO_URI` accordingly |
| MongoDB connection refused in seed script | Stack may still be starting — wait for healthcheck (`docker compose ps`), then re-run seed |
| Charts show empty / no data | Run `python scripts/seed_mongo.py`; verify CSV at `data/processed/analytics/products_raw.csv` |
| Image build fails | Clear cached layers: `docker compose build --no-cache` |
| `gunicorn: command not found` | Confirm gunicorn is in `requirements.txt`: `grep gunicorn requirements.txt` |

## Part 10 — Assignment 13 Solution Summary

### What was built

| Component | File | Deliverable |
|---|---|---|
| Dash Application (35%) | `app.py`, `src/dashboard/layout.py` | Full dark-themed layout with 3 interactive controls and 4+1 charts |
| Callbacks (25%) | `src/dashboard/callbacks.py` | 5 callbacks (4 filter + 1 live ticker with `dcc.Interval`) |
| MongoDB Integration (15%) | `src/dashboard/data_access.py`, `scripts/seed_mongo.py` | Reads from MongoDB, falls back to CSV gracefully |
| Docker Deployment (15%) | `Dockerfile`, `docker-compose.yml`, `.dockerignore` | Single `docker compose up --build` starts full stack |
| Deployment Documentation (10%) | This notebook (Part 9 above) | Prerequisites, step-by-step commands, verification checklist, troubleshooting table |

### Visualisation design decisions (Tufte principles)

- **Horizontal bar chart for top products** — long product titles fit naturally on the horizontal axis, avoiding diagonal label rotation that hides data.
- **Box plot for ratings by source** — shows median, interquartile range, and outliers simultaneously. A simple bar of means would hide the variance that reveals the real story about review quality across sources.
- **Histogram for rating distribution** — reveals the skew toward 5-star ratings (positivity bias), which would be invisible in a box plot or bar chart.
- **Bar chart for products per source** — sources are discrete unordered categories; bars communicate independence correctly, unlike a line which implies continuity.
- **Dark theme throughout** — consistent `plotly_dark` template with `#112236` panel backgrounds creates a cohesive dashboard feel and reduces eye strain.

### Key architectural decisions

- **`server = app.server`** in `app.py` exposes the underlying Flask application so Gunicorn can serve it in production without any code change.
- **Environment variable for `MONGO_URI`** — the same application binary works locally (defaults to `localhost:27017`) and inside Docker Compose (receives `mongodb://db:27017` from `docker-compose.yml`).
- **`collections.deque(maxlen=60)`** for the live ticker — a fixed-capacity circular buffer guarantees constant memory regardless of how long the dashboard runs.
- **CSV fallback in `load_products_df()`** — if MongoDB is unavailable the application degrades gracefully and still shows all four filter-driven charts.